In [ ]:
!pip install torch==2.6.0 --quiet
!pip install pandas==2.2.3 --quiet
!pip install matplotlib==3.10.1 --quiet
import torch
import torch.nn as nn
import torch.optim as optim
import time
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
GLOBAL_SEED = 5 #изменять этот seed
torch.manual_seed(GLOBAL_SEED)
torch.set_num_threads(4)

ds = [2, 5, 10, 20]
ms = [32, 64, 128, 256, 512, 1024]
n_train_base = 50000
n_test = 10000
batch_size = 512
weight_decay = 1e-4
max_epochs_base = 10000

lr_dict = {
    "Radial": 1e-3,
    "Ridge sum": 1e-3,
    "Fourier narrow": 1e-3,
    "Fourier noisy": 1e-2
}

def f_radial(x):
    c = 0.5 * torch.ones(x.shape[1])
    return torch.exp(-0.5 * torch.sum((x - c)**2, dim=1))

def f_ridge_sum(x):
    d = x.shape[1]
    a1 = torch.tensor([1,1]+[0.5]*(d-2), dtype=torch.float32)[:d]
    a2 = torch.tensor([0.5,1,1]+[0.5]*(d-3), dtype=torch.float32)[:d]
    a3 = torch.tensor([1,0.5,1,1]+[1]*(d-4), dtype=torch.float32)[:d]
    a1, a2, a3 = a1/a1.norm(), a2/a2.norm(), a3/a3.norm()
    b1, b2, b3 = 0.1, 0.7, 0.3
    return torch.tanh(x@a1+b1) + torch.sin(x@a2+b2) + torch.cos(x@a3+b3)

def f_fourier_narrow(x):
    d = x.shape[1]
    w1 = torch.ones(d)
    w2 = torch.tensor([2,1.5]+[1]*(d-2))
    return torch.cos(x@w1) + 0.5*torch.sin(x@w1) + 0.8*torch.cos(x@w2) - 0.3*torch.sin(x@w2)

def f_fourier_noisy(x):
    d = x.shape[1]
    w1 = torch.ones(d)
    w2 = torch.ones(d)*10
    return torch.cos(x@w1) + 0.5*torch.sin(x@w1) + 0.8*torch.cos(x@w2) - 0.3*torch.sin(x@w2)

funcs = [f_radial, f_ridge_sum, f_fourier_narrow, f_fourier_noisy]
names = ["Radial", "Ridge sum", "Fourier narrow", "Fourier noisy"]

results = []

for d in ds:
    gen_data = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d)
    gen_batch = torch.Generator().manual_seed(GLOBAL_SEED + 1000*d + 1)
    n_train = n_train_base * d

    x_train = (torch.rand(n_train, d, generator=gen_data)-0.5)*2
    x_test  = (torch.rand(n_test, d, generator=gen_data)-0.5)*2

    for f, name in zip(funcs, names):
        y_train = f(x_train)
        y_test  = f(x_test)

        for m in ms:
            model = nn.Sequential(nn.Linear(d, m), nn.Tanh(), nn.Linear(m, 1))

            base_lr = lr_dict[name]
            lr_m = base_lr * (32/m)**0.25

            optimizer = optim.Adam(model.parameters(), lr=lr_m, weight_decay=weight_decay)
            loss_fn = nn.MSELoss()

            num_epochs = max_epochs_base + 5000*(m//256)
            steps_per_epoch = n_train // batch_size
            total_steps = num_epochs * steps_per_epoch

            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=lr_m, total_steps=total_steps, pct_start=0.1,
                anneal_strategy='cos', div_factor=25.0, final_div_factor=100.0
            )

            start_time = time.time()
            for epoch in range(num_epochs):
                idx = torch.randint(0, n_train, (batch_size,), generator=gen_batch)
                xb, yb = x_train[idx], y_train[idx].unsqueeze(1)

                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            train_time = time.time() - start_time

            with torch.no_grad():
                y_pred = model(x_test)
                l2_error = torch.sqrt(torch.mean((y_pred - y_test.unsqueeze(1))**2)).item()

            results.append({"function": name, "d": d, "m": m,
                            "L2_error": l2_error, "train_time_sec": train_time})
            print(f"{name:14} d={d:2d}, m={m:4d}, L2={l2_error:.6f}, time={train_time:.1f}s, lr={lr_m:.6f}")

df = pd.DataFrame(results)
functions = df['function'].unique()
colors = ['r', 'g', 'b', 'm', 'c', 'y']

for func in functions:
    plt.figure(figsize=(8,5))
    df_func = df[df['function'] == func]

    for i, d in enumerate(sorted(df_func['d'].unique())):
        df_plot = df_func[df_func['d'] == d].sort_values('m')
        plt.plot(df_plot['m'], df_plot['L2_error'], marker='o', color=colors[i], label=f'd={d}')

    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("m (ширина сети)")
    plt.ylabel("L2 ошибка")
    plt.title(f"L2 ошибка vs m для функции '{func}'")
    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.6)
    plt.show()